# Co-Re Grippers

The Co-Re grippers mount on pipetting channels and allow for moving labware around the deck. This tutorial shows how to use them with the {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` device.

## Gripper types

There are two different types of Co-Re grippers:

![Co-Re gripper types](img/core-grippers/core-gripper-types.jpg)

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` and create the device. The deck is available as `star.deck`.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR

star = STAR()
await star.setup()

Set up a plate carrier and plate for the examples below.

In [ ]:
from pylabrobot.resources import PLT_CAR_L5AC_A00, CellTreat_96_wellplate_350ul_Ub

plate_carrier = PLT_CAR_L5AC_A00(name="plate_carrier")
plate_carrier[0] = plate = CellTreat_96_wellplate_350ul_Ub(name="plate")
star.deck.assign_child_resource(plate_carrier, rails=14)

## Moving resources

Use {meth}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR.core_grippers` as an async context manager. It picks up the gripper tools when entering and returns them when exiting. The yielded {class}`~pylabrobot.capabilities.arms.arm.GripperArm` has two APIs:

- `move_resource`: a single call that picks up a resource and drops it at a destination.
- `pick_up_resource` and `drop_resource`: two separate calls for more control over timing.

### `move_resource`

This is the simplest way to move a plate. Specify `front_channel` to choose which pair of pipetting channels picks up the gripper tools (the back channel is `front_channel - 1`).

In [ ]:
async with star.core_grippers(front_channel=7) as arm:
    await arm.move_resource(plate, plate_carrier[1], pickup_distance_from_bottom=10)

### `pick_up_resource` and `drop_resource`

For more control over the pick-up and drop actions, use them separately within the same context manager.

In [ ]:
async with star.core_grippers(front_channel=7) as arm:
    await arm.pick_up_resource(plate, pickup_distance_from_bottom=10)
    # ... do something between pick-up and drop ...
    await arm.drop_resource(plate_carrier[0])

The gripper tools are automatically returned to their parking position when the `async with` block exits, so you never need to manage that manually.

## Teardown

In [ ]:
await star.stop()